# Combined Amber RAG (ChromaDB archive + PDF FAISS) — POC

Single query → hits **two** vector stores in parallel, merges results by similarity score, keeps the top 5 overall, feeds them to the LLM.

- **Store A** — persistent **Chroma** at `/opt/chromadb/data/prompt_db`.
- **Store B** — in-memory **FAISS** built once from `Amber25.pdf`, cached to `./faiss_pdf_index/` so it loads instantly on subsequent runs.

Both stores use `sentence-transformers/all-MiniLM-L6-v2` and L2 (squared) distance, so scores are directly comparable across stores.

## 1) Setup

In [127]:
import os
from typing import List, Tuple

import chromadb
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_ollama import ChatOllama

## 2) Shared embedding model
Both stores must use the same embedding model for the scores to be comparable.

In [128]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## 3) Open both stores

- **Archive store** — persistent Chroma at `/opt/chromadb/data/prompt_db`.
- **PDF store** — in-memory FAISS. On first run, it parses `Amber25.pdf`, embeds the chunks, and saves a FAISS index to `./faiss_pdf_index/`. On every subsequent run, it loads that cached index in < 1s instead of re-embedding ~3,800 chunks.

Delete `./faiss_pdf_index/` to force a rebuild (e.g. when the PDF or chunking settings change).

In [129]:
# --- Store A: archive / emails (persistent Chroma) ---
ARCHIVE_DB_PATH = "/opt/chromadb/data/prompt_db"
archive_client = chromadb.PersistentClient(path=ARCHIVE_DB_PATH)
vectorstore_archive = Chroma(
    client=archive_client,
    collection_name="amber_messages",
    embedding_function=embeddings,
)

# --- Store B: PDF (in-memory FAISS with cached index on disk) ---
PDF_PATH = "Amber25.pdf"
FAISS_INDEX_DIR = "./faiss_pdf_index"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100


def _clean_pdf_text(text: str) -> str:
    text = " ".join(text.split())
    text = text.replace("\ufb01", "fi").replace("\ufb02", "fl")
    return text


def _build_pdf_chunks(pdf_path: str) -> List[Document]:
    pages = PyPDFLoader(pdf_path).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=[" "]
    )
    chunks: List[Document] = []
    for page_num, page in enumerate(pages):
        cleaned = _clean_pdf_text(page.page_content)
        if len(cleaned.strip()) < 50:
            continue
        chunks.extend(splitter.create_documents(
            texts=[cleaned],
            metadatas=[{
                **page.metadata,
                "page": page_num + 1,
                "total_pages": len(pages),
                "chunk_method": "smart_pdf_processor",
                "char_count": len(cleaned),
            }],
        ))
    return chunks


if os.path.isdir(FAISS_INDEX_DIR):
    vectorstore_pdf = FAISS.load_local(
        FAISS_INDEX_DIR, embeddings, allow_dangerous_deserialization=True
    )
    print(f"Loaded cached FAISS index from {FAISS_INDEX_DIR}")
else:
    print(f"No cached index at {FAISS_INDEX_DIR} — building from {PDF_PATH} ...")
    pdf_chunks = _build_pdf_chunks(PDF_PATH)
    print(f"  {len(pdf_chunks)} chunks, embedding ...")
    vectorstore_pdf = FAISS.from_documents(pdf_chunks, embeddings)
    vectorstore_pdf.save_local(FAISS_INDEX_DIR)
    print(f"  Saved FAISS index to {FAISS_INDEX_DIR}")

print(f"Archive store: {vectorstore_archive._collection.count()} vectors @ {ARCHIVE_DB_PATH}")
print(f"PDF store:     {vectorstore_pdf.index.ntotal} vectors (FAISS in-memory)")

Loaded cached FAISS index from ./faiss_pdf_index
Archive store: 42935 vectors @ /opt/chromadb/data/prompt_db
PDF store:     3793 vectors (FAISS in-memory)


## 4) Hybrid retriever — merge both stores, keep top 5
Each store is queried for its own top-K. We tag every doc with which store it came from, sort by raw L2 distance (lower = closer), and keep the best `k_total` overall.

In [130]:
def hybrid_retrieve(
    query: str,
    k_total: int = 5,
    k_archive: int = 10,
    k_pdf: int = 10,
) -> List[Document]:
    """Query both stores, merge by distance (lower = closer), return top k_total docs."""
    scored: List[Tuple[Document, float]] = []

    for doc, score in vectorstore_archive.similarity_search_with_score(query, k=k_archive):
        doc.metadata = {**doc.metadata, "store": "archive", "score": float(score)}
        scored.append((doc, score))

    for doc, score in vectorstore_pdf.similarity_search_with_score(query, k=k_pdf):
        doc.metadata = {**doc.metadata, "store": "pdf", "score": float(score)}
        scored.append((doc, score))

    scored.sort(key=lambda pair: pair[1])  # L2: lower is closer
    return [doc for doc, _ in scored[:k_total]]

# Quick smoke test
hits = hybrid_retrieve("What is Amber?", k_total=5)
for i, d in enumerate(hits, 1):
    print(f"{i}. [{d.metadata.get('store')}] score={d.metadata.get('score'):.4f} "
          f"page={d.metadata.get('page', '-')}")
    print("   ", d.page_content[:160].replace("\n", " "), "...")


1. [archive] score=0.3206 page=-
    Steve Seibold < seibold.chemistry.msu.edu > (Wed, 5 May 2010 08:38:46 -0400): Hi Sorry if this is an ignorant question, but I have been reading on the Amber ema ...
2. [archive] score=0.3282 page=-
    via AMBER < amber.ambermd.org > (Mon, 12 Jun 2023 02:03:48 +0800 (CST)):  ...
3. [archive] score=0.3356 page=-
    < steinbrt.rci.rutgers.edu > (Thu, 28 Mar 2013 09:16:56 -0400 (EDT)): Hi,  > Thank you very much.. I am very new to amber. First time i am using&nbsp;  > follow ...
4. [archive] score=0.3400 page=-
    via AMBER < amber.ambermd.org > (Mon, 1 Jan 2024 00:25:24 +0800 (CST)):  ...
5. [archive] score=0.3631 page=-
    Carlos Simmerling < carlos.csb.sunysb.edu > (Fri, 15 Sep 2006 15:47:34 -0400): I apologize for using the Amber list to advertise my own article, but I think it  ...


## 5) LLM + prompt

In [131]:
llm = ChatOllama(model="llama3.1:8b", temperature=0)

In [132]:
custom_prompt = ChatPromptTemplate.from_template("""You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).

CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention any Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output the final answer.
For source citations, include the source page number (for PDF) or source label (for archive) at the beginning.

Context:
{context}

Question: {question}

Answer:""")

## 6) LCEL chain using the hybrid retriever

In [133]:
def format_docs(docs: List[Document]) -> str:
    formatted = []
    for i, doc in enumerate(docs, 1):
        store = doc.metadata.get("store", "unknown")
        source = doc.metadata.get("source", "unknown_source")
        page = doc.metadata.get("page", "-")
        score = doc.metadata.get("score", None)
        score_str = f" | Score: {score:.4f}" if isinstance(score, (int, float)) else ""
        formatted.append(
            f"[Chunk {i} | Store: {store} | Source: {source} | Page: {page}{score_str}]\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(formatted)

In [134]:
# Wrap the hybrid retrieval in a RunnableLambda so it plugs into LCEL like any other retriever.
hybrid_retriever = RunnableLambda(lambda q: hybrid_retrieve(q, k_total=5))

rag_chain_lcel = (
    {
        "context": hybrid_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

## 7) Query helper — answer + the top-5 sources that fed it

In [135]:
def query_rag(question: str, k_total: int = 5):
    print(f"Question: {question}")
    print("-" * 60)

    docs = hybrid_retrieve(question, k_total=k_total)
    answer = rag_chain_lcel.invoke(question)

    print("Answer:")
    print(answer)

    print("\nTop sources used:")
    for i, doc in enumerate(docs, 1):
        store = doc.metadata.get("store", "?")
        page = doc.metadata.get("page", "-")
        src = doc.metadata.get("source", "-")
        score = doc.metadata.get("score")
        score_str = f"{score:.4f}" if isinstance(score, (int, float)) else "-"
        print(f"\n--- Source {i} [{store}] page={page} src={src} score={score_str} ---")
        print(doc.page_content[:240].replace("\n", " "), "...")

    return answer, docs

## 8) Try it

In [136]:
_ = query_rag("What is Amber?")

Question: What is Amber?
------------------------------------------------------------
Answer:
http://ambermd.org/pmwiki/ (Source: Carlos Simmerling's response)

The AMBER molecular dynamics suite and its associated tools are a collection of software packages used for simulating the behavior of molecules, particularly proteins and nucleic acids. The AMBER force field is a set of parameters that describe the interactions between atoms in a molecule.

Technical Explanation:
AMBER (Assisted Model Building with Energy Refinement) is a molecular dynamics simulation package developed by Dr. Peter Kollman's group at the University of California, San Francisco. It uses a combination of classical mechanics and quantum mechanics to simulate the behavior of molecules over time. The AMBER force field is a set of parameters that describe the interactions between atoms in a molecule, including bond stretching, angle bending, dihedral angles, and non-bonded interactions.

Practical Guidance:
To get st

In [137]:
_ = query_rag("Why should SHAKE be disabled during minimization in AMBER?")

Question: Why should SHAKE be disabled during minimization in AMBER?
------------------------------------------------------------
Answer:
Since SHAKE is an algorithm based on dynamics, the minimizer is not aware of what SHAKE is doing; for this reason, minimizations generally should be carried out without SHAKE. (Source: Page 329 of AMBER18 manual)

Technical Explanation:
The minimizer in AMBER does not have knowledge about the constraints applied by SHAKE during dynamics simulations. Therefore, it's recommended to disable SHAKE during minimization steps.

Practical Guidance:
To disable SHAKE during minimization, set `ntc=1` or `ntc=2` depending on your specific needs. However, as mentioned in the context, using `ntc=2` at the initial step may solve the problem for some cases, such as with rigid water models like TIP4P-D.

Output:
The minimizer should be run without SHAKE to ensure accurate results.

Top sources used:

--- Source 1 [archive] page=- src=- score=0.2841 ---
case < case.bi

In [138]:
_ = query_rag("How do I obtain a Z-DNA structure from NAB?")

Question: How do I obtain a Z-DNA structure from NAB?
------------------------------------------------------------
Answer:
You can use the `fd_helix()` routine in NAB to create a variety of DNA helices, including Z-DNA. Alternatively, you can visit http://w3dna.rutgers.edu/index.php/rebuild for more options.

Technical Explanation:
The `fd_helix()` function is used to create a DNA helix with specified parameters. For creating Z-DNA, you would need to specify the correct parameters such as the pitch and twist of the helix. The `http://w3dna.rutgers.edu/index.php/rebuild` website provides a more user-friendly interface for building various types of DNA structures, including Z-DNA.

Practical Guidance:
To create a Z-DNA structure using NAB, you would need to specify the correct parameters in the `fd_helix()` function. This may involve consulting the documentation and examples provided with NAB or seeking guidance from experienced users. Alternatively, visiting http://w3dna.rutgers.edu/ind

In [139]:
_ = query_rag("How can I get SHAKE to consider two different residue names to be water?")

Question: How can I get SHAKE to consider two different residue names to be water?
------------------------------------------------------------
Answer:
You might be able to escape SETTLE by renaming water residues. Sander/pmemd are not actually that smart and just find water by being told that the residue name is "WAT" (by default). If you have some other name you'll just get regular SHAKE I think, or nothing at all with the right choice of ntc/ntf.

Technical Explanation:
The AMBER force field parameters are defined for specific atom types and residue names. By default, pmemd uses the "WAT" residue name to identify water molecules. If you rename water residues to something else, such as "CRY" or "WAT2", pmemd will not recognize them as water and will apply SHAKE constraints instead of SETTLE.

Practical Guidance:
To get SHAKE to consider two different residue names to be water, you can use the "shakemask" variable to specify which atoms should be shaken. However, this requires a time 

In [140]:
_ = query_rag("Best OS recommendation for installing Amber + CUDA on an RTX 3070 server: Rocky Linux vs Ubuntu? What setup is known to work?")

Question: Best OS recommendation for installing Amber + CUDA on an RTX 3070 server: Rocky Linux vs Ubuntu? What setup is known to work?
------------------------------------------------------------
Answer:
**Ubuntu 22.04 LTS with CUDA 12.3 is recommended**

According to Todd Minehardt's response, Ubuntu 22.04 LTS with AmberTools23 and CUDA 12.3 works fine for installing Amber and Schrodinger licenses together on an RTX 3070 server.

Technical Explanation:
The compatibility of the OS and CUDA version is crucial for running Amber simulations efficiently. The recommended setup ensures that the operating system, CUDA version, and drivers are all compatible, which can be checked on the NVIDIA website for full details.

Practical Guidance:
To install Amber and Schrodinger licenses together on an RTX 3070 server, Rufaida Al Zoubi is advised to switch to Ubuntu 22.04 LTS with CUDA 12.3. This setup has been confirmed to work by Todd Minehardt, and it's essential to make some changes to the insta

In [141]:
_ = query_rag("How do I use paramfit to generate force field parameters for boron-containing compounds?")

Question: How do I use paramfit to generate force field parameters for boron-containing compounds?
------------------------------------------------------------
Answer:
**Search the archive for relevant posts**

To generate force field parameters for boron-containing compounds using paramfit, search the Amber archive for relevant posts. Carlos Simmerling suggests searching the archive with "boron" in the search bar (Chunk 1).

**No direct answer to paramfit usage**

There is no direct answer to how to use paramfit to generate force field parameters for boron-containing compounds in the provided context.

**David Case's advice on literature search and mgdx/parmfit**

However, David Case suggests that a literature search might help simplify the process of developing new force field parameters (Chunk 4). He also mentions using mgdx or parmfit to generate a force field by hand, possibly with quantum calculations as input (Chunk 4).

**Technical explanation**

The AMBER force field does not 

In [142]:
_ = query_rag("I have run a short TI simulation on a system using pmemd wherein, I have in one state bonded disulphide bridge (state A) and in the other unbonded bridge (State B). The peptide I am working with is 40 residues long. Can somebody kindly suggest me how do I extract pdb of the two states?")

Question: I have run a short TI simulation on a system using pmemd wherein, I have in one state bonded disulphide bridge (state A) and in the other unbonded bridge (State B). The peptide I am working with is 40 residues long. Can somebody kindly suggest me how do I extract pdb of the two states?
------------------------------------------------------------
Answer:
To extract PDB files for both states, you can use cpptraj as suggested by Hannes Loeffler. You will need to adjust the strip mask corresponding to your respective states.

For state A (bonded disulphide bridge), use the following command:

cpptraj -p *.parm7 <<_EOF
trajin ${s}_prepare/press.rst7
strip ":1-40"
outtraj ${s}_stateA.pdb onlyframes 1
_EOF

For state B (unbonded disulphide bridge), use the following command:

cpptraj -p *.parm7 <<_EOF
trajin ${s}_prepare/press.rst7
strip ":1-3,5-29,31-40"
outtraj ${s}_stateB.pdb onlyframes 1
_EOF

This will generate two PDB files, one for each state.

Technical Explanation:
The stri

In [143]:
_ = query_rag("""
I'm using CPPTRAJ to analyze simulations of parallel strand DNA and want to
> get step parameters via nastruct. Does anyone know how nastruct identify
> the molecule as parallel strands? If I write "guessbp bptype para" in the
> nastruct command, the program will simply get stuck and look as if it is
> not continuing to run at all. If I just write "guessbp", it would seem to
> treat the molecule as anti-parallel. What is the correct way to do that?
""")

Question: 
I'm using CPPTRAJ to analyze simulations of parallel strand DNA and want to
> get step parameters via nastruct. Does anyone know how nastruct identify
> the molecule as parallel strands? If I write "guessbp bptype para" in the
> nastruct command, the program will simply get stuck and look as if it is
> not continuing to run at all. If I just write "guessbp", it would seem to
> treat the molecule as anti-parallel. What is the correct way to do that?

------------------------------------------------------------
Answer:
**Direct Answer**
You should upgrade your cpptraj version to 6.18.0 or later, as base pair detection has been greatly improved since then.

**Technical Explanation**
The `nastruct` command in CPPTRAJ uses algorithms and base reference frames similar to those used by 3DNA, but it does not calculate global helical axis parameters for single-strand RNA. However, local helical axis parameters for base pair steps are determined.

**Practical Guidance**
To upgrade you